In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))

def permute_rnd_colors_(query):
    permutation = np.random.permutation(10).tolist()
    return 'colors' + ''.join(map(str, permutation))

def flip_lr(grid):
    return np.fliplr(np.asarray(grid))

def flip_ud(grid):
    return np.flipud(np.asarray(grid))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True, use_colors=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op=='flip_lr':            a = np.fliplr(a)
            elif op=='flip_ud':            a = np.flipud(a)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('colors'):  a = permute_mod(a, op, invert=False) if use_colors else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True, inv_colors=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op=='flip_lr':            a = np.fliplr(a)
            elif op=='flip_ud':            a = np.flipud(a)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('colors'):  a = permute_mod(a, op, invert=True) if inv_colors else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def augment_v2(self, n=1, shfl_keys=False, seed=42):
        """Richer augmentation with flips and color remapping for rescue passes.

        Uses fewer permute repeats to keep total sample count similar to augment(),
        but adds geometric flips and independent color-permutation branches that
        cover transformation types the default pipeline misses.
        """
        np.random.seed(seed)
        n_small = max(1, n // 4)
        n_perm = max(1, n_small // 2)
        n_colors = max(1, n_small - n_perm)

        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(flip_lr, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n_perm, shuffle=shfl_keys, keep=False)
        d = d.mod(permute_mod, permute_rnd_colors_, n=n_colors, shuffle=shfl_keys, keep=True)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def _safe_mean(values):
    return float(np.mean(values)) if len(values) else 0.0


def _grid_signature(guess):
    arr = np.asarray(guess)
    shape = tuple(arr.shape)
    flat = tuple(arr.reshape(-1).tolist())
    return shape, flat


def _is_base_guess(sample):
    source = f"{sample.get('_source_key', '')}{sample.get('_source_run', '')}"
    return ".base." in source or ".base.run" in source or source.endswith(".base")


def _unique_solution_count(values):
    return len({hashable(sample["solution"]) for sample in values.values()})

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    ranked = []
    for sc, output in scores.values():
        metric = float(getter(sc))
        support = len(sc)
        beam_best = float(np.min([g["beam_score"] for g in sc])) if support else float("inf")
        aug_mean = _safe_mean([_safe_mean(g["score_aug"]) for g in sc])
        # Deterministic tie-breaks reduce submission variance from file iteration order.
        ranked.append((metric, support, beam_best, aug_mean, _grid_signature(output), output))
    ranked = sorted(ranked, key=lambda x: (-x[0], -x[1], x[2], x[3], x[4]))
    ordered_outputs = [x[-1] for x in ranked]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)

def getter_hybrid_consensus(guesses):
    support = len(guesses)
    beam_scores = [g["beam_score"] for g in guesses]
    aug_scores = [_safe_mean(g["score_aug"]) for g in guesses]
    beam_best = float(np.min(beam_scores))
    beam_mean = _safe_mean(beam_scores)
    aug_mean = _safe_mean(aug_scores)
    aug_std = float(np.std(aug_scores)) if len(aug_scores) else 0.0
    return (2.5 * support) - (1.2 * beam_best) - (0.6 * beam_mean) - (0.8 * aug_mean) - (0.1 * aug_std)

def score_hybrid_consensus(guesses):
    return score_sum(guesses, getter_hybrid_consensus)

def getter_round_consensus(guesses):
    support = len(guesses)
    unique_rounds = len({g.get("_round", -1) for g in guesses})
    unique_runs = len({g.get("_source_run", "") for g in guesses})
    beam_scores = [g["beam_score"] for g in guesses]
    aug_scores = [_safe_mean(g["score_aug"]) for g in guesses]
    beam_best = float(np.min(beam_scores))
    beam_mean = _safe_mean(beam_scores)
    aug_mean = _safe_mean(aug_scores)
    return (2.8 * unique_rounds) + (1.4 * unique_runs) + (1.2 * support) - (1.0 * beam_best) - (0.5 * beam_mean) - (0.8 * aug_mean)

def score_round_consensus(guesses):
    return score_sum(guesses, getter_round_consensus)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
    score_hybrid_consensus,
    score_round_consensus,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        loaded_files = 0
        for key in sorted(os.listdir(store)):
            path = os.path.join(store, key)
            if not os.path.isfile(path):
                continue
            try:
                with bz2.BZ2File(path) as f:
                    outputs = pickle.load(f)
            except Exception:
                # Ignore non-bz2 / unreadable files in the output directory.
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                enriched = dict(sample)
                enriched["_source_key"] = key
                enriched["_source_run"] = run_name or ".run0"
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = enriched
            loaded_files += 1
        print(f"*** Loaded {loaded_files} decoded files from '{store}'")

    @staticmethod
    def merge_rankings(rankings):
        merged = {}
        for ranking in rankings:
            n = max(1, len(ranking))
            for rank, guess in enumerate(ranking):
                h = hashable(guess)
                item = merged.get(h, {"guess": guess, "score": 0.0, "best_rank": rank})
                item["score"] += (n - rank)
                item["best_rank"] = min(item["best_rank"], rank)
                merged[h] = item
        ordered = sorted(merged.values(), key=lambda x: (x["score"], -x["best_rank"]), reverse=True)
        return [x["guess"] for x in ordered]

    def run_selection_ensemble(self, selection_algorithms_local=None):
        algos = selection_algorithms if selection_algorithms_local is None else selection_algorithms_local
        selected = {}
        for bk, values in self.decoded_results.items():
            guesses = {k: g for k, g in values.items()}
            rankings = [algo(guesses) for algo in algos]
            selected[bk] = self.merge_rankings(rankings)
        return selected

    def run_selection_dual(self, primary_algorithm=score_kgmon, secondary_algorithm=score_full_probmul_3):
        selected = {}
        for bk, values in self.decoded_results.items():
            guesses = {k: g for k, g in values.items()}
            primary = primary_algorithm(guesses)
            secondary = secondary_algorithm(guesses)
            merged = []
            seen = set()

            def append_unique(candidates):
                for guess in candidates:
                    h = hashable(guess)
                    if h not in seen:
                        seen.add(h)
                        merged.append(guess)

            if primary:
                append_unique([primary[0]])
            if secondary:
                append_unique([secondary[0]])
            append_unique(primary[1:])
            append_unique(secondary[1:])
            selected[bk] = merged
        return selected

    def run_selection_dual_round(self):
        return self.run_selection_dual(primary_algorithm=score_round_consensus, secondary_algorithm=score_kgmon)

    def run_selection_guarded_dual(
        self,
        primary_algorithm=score_kgmon,
        secondary_algorithm=score_full_probmul_3,
        min_base_candidates=1,
        min_base_unique=1,
    ):
        selected = {}
        for bk, values in self.decoded_results.items():
            all_guesses = {k: g for k, g in values.items()}
            base_guesses = {k: g for k, g in all_guesses.items() if _is_base_guess(g)}
            use_base_primary = (
                len(base_guesses) >= min_base_candidates
                and _unique_solution_count(base_guesses) >= min_base_unique
            )
            primary_guesses = base_guesses if use_base_primary else all_guesses

            primary = primary_algorithm(primary_guesses)
            secondary = secondary_algorithm(all_guesses)
            merged = []
            seen = set()

            def append_unique(candidates, max_new=None):
                added = 0
                for guess in candidates:
                    h = hashable(guess)
                    if h in seen:
                        continue
                    seen.add(h)
                    merged.append(guess)
                    added += 1
                    if max_new is not None and added >= max_new:
                        break

            # Attempt 1 stays anchored to kgmon, preferably using the base pass only.
            append_unique(primary[:1])
            # Attempt 2 is the best full-probability alternative, scanning past duplicates.
            append_unique(secondary, max_new=1)
            append_unique(primary[1:])
            append_unique(secondary)
            selected[bk] = merged
        return selected

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_selection_diverse_dual(
        self,
        primary_algorithm=score_kgmon,
        secondary_algorithm=score_full_probmul_3,
        min_base_candidates=1,
        min_base_unique=1,
    ):
        """Like guarded_dual but attempt_2 prefers solutions from diverse augmentation sources.

        When multiple candidates have similar scores, the one that appears across more
        distinct geometric transformations (rot90, transpose, flip_lr, ...) is preferred.
        """
        selected = {}
        for bk, values in self.decoded_results.items():
            all_guesses = {k: g for k, g in values.items()}
            base_guesses = {k: g for k, g in all_guesses.items() if _is_base_guess(g)}
            use_base_primary = (
                len(base_guesses) >= min_base_candidates
                and _unique_solution_count(base_guesses) >= min_base_unique
            )
            primary_guesses = base_guesses if use_base_primary else all_guesses

            primary = primary_algorithm(primary_guesses)

            # Build diversity-augmented scoring for secondary ranking
            by_solution = {}
            for k, g in all_guesses.items():
                h = hashable(g["solution"])
                entry = by_solution.get(h, {"samples": [], "solution": g["solution"]})
                entry["samples"].append(g)
                # Extract transformation family from source key
                source = g.get("_source_key", "")
                for op in source.split("."):
                    if op in ("transpose", "rot90", "flip_lr", "flip_ud", "permute", "colors"):
                        entry.setdefault("ops", set()).add(op.split("0")[0] if op.startswith("rot") else op)
                by_solution[h] = entry

            # Diversity-aware secondary ordering
            diverse_ranked = []
            for h, entry in by_solution.items():
                samples = entry["samples"]
                solution = entry["solution"]
                n_sources = len(entry.get("ops", set()))
                support = len(samples)
                beam_best = float(np.min([g["beam_score"] for g in samples])) if support else float("inf")
                aug_mean = _safe_mean([_safe_mean(g["score_aug"]) for g in samples])
                # Score: full_probmul_3 metric + diversity bonus from distinct augmentation ops
                probmul_score = getter_full_probmul_3(samples)
                diversity_bonus = n_sources * 0.5
                metric = probmul_score + diversity_bonus
                diverse_ranked.append((metric, support, beam_best, aug_mean, _grid_signature(solution), solution))
            diverse_ranked = sorted(diverse_ranked, key=lambda x: (-x[0], -x[1], x[2], x[3], x[4]))
            secondary = [x[-1] for x in diverse_ranked]

            merged = []
            seen = set()

            def append_unique(candidates, max_new=None):
                added = 0
                for guess in candidates:
                    h = hashable(guess)
                    if h in seen:
                        continue
                    seen.add(h)
                    merged.append(guess)
                    added += 1
                    if max_new is not None and added >= max_new:
                        break

            append_unique(primary[:1])
            append_unique(secondary, max_new=1)
            append_unique(primary[1:])
            append_unique(secondary)
            selected[bk] = merged
        return selected

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        if num_total_keys == 0:
            print(" subkeys: 0/0")
            print(" avg correct beam score:      nan")
            print(" max correct beam score:      nan")
            print(" [WARN] No decoded outputs loaded; skip benchmark.")
            return

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if len(correct_beam_scores):
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg correct beam score:      nan")
            print(" max correct beam score:      nan")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

        selected_ensemble = self.run_selection_ensemble()
        correct_puzzles = {k for k, v in selected_ensemble.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
        score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
        print(f" acc: {score:5.1f}/{num_puzzles:3} ('ensemble')")

        selected_dual = self.run_selection_dual()
        correct_puzzles = {k for k, v in selected_dual.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
        score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
        print(f" acc: {score:5.1f}/{num_puzzles:3} ('dual')")

        selected_guarded_dual = self.run_selection_guarded_dual()
        correct_puzzles = {k for k, v in selected_guarded_dual.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
        score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
        print(f" acc: {score:5.1f}/{num_puzzles:3} ('guarded_dual')")

        selected_diverse_dual = self.run_selection_diverse_dual()
        correct_puzzles = {k for k, v in selected_diverse_dual.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
        score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
        print(f" acc: {score:5.1f}/{num_puzzles:3} ('diverse_dual')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr
import hashlib

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "膴": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def env_int(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        return default


def env_float(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return float(value)
    except ValueError:
        return default


def env_str(name, default):
    value = os.getenv(name)
    return default if value is None or value == "" else value


def env_bool(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_int_list(raw, default):
    if raw is None or raw.strip() == "":
        raw = default
    result = []
    for item in raw.split(","):
        item = item.strip()
        if not item:
            continue
        result.append(int(item))
    return result


def parse_float_list(raw, default):
    if raw is None or raw.strip() == "":
        raw = default
    result = []
    for item in raw.split(","):
        item = item.strip()
        if not item:
            continue
        result.append(float(item))
    return result


def list_value(values, idx):
    if not values:
        return None
    if idx < len(values):
        return values[idx]
    return values[-1]


def get_queue_size(queue):
    try:
        return int(queue.qsize())
    except Exception:
        return -1


def stable_seed_from_key(key, mod=1024**2):
    if key is None:
        return 0
    digest = hashlib.blake2b(str(key).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "little") % mod


def compute_puzzle_budget(
    budget_mode,
    remaining_global_time,
    pending_after_pop,
    max_puzzle_seconds,
    min_puzzle_seconds,
    world_size,
):
    if pending_after_pop < 0:
        return max_puzzle_seconds
    tasks_left = max(1, pending_after_pop + 1)
    if budget_mode == "adaptive_parallel":
        adaptive_budget = remaining_global_time * max(1, world_size) / tasks_left
        return min(max_puzzle_seconds, max(min_puzzle_seconds, adaptive_budget * 1.1))
    if budget_mode == "adaptive_serial":
        adaptive_budget = remaining_global_time / tasks_left
        return min(max_puzzle_seconds, max(min_puzzle_seconds, adaptive_budget * 1.25))
    return max_puzzle_seconds


def build_decode_batches(subkeys):
    if len(subkeys) == 16:
        batches = []
        for offsets in ([0, 4], [2, 6], [8, 12], [10, 14]):
            batch = []
            for offset in offsets:
                batch.extend(subkeys[offset:offset+2])
            if len(batch):
                batches.append(batch)
        return batches
    return [subkeys[i:i+4] for i in range(0, len(subkeys), 4) if subkeys[i:i+4]]


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 馃敡 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_time_limit) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < dfs_time_limit and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_time_limit=dfs_time_limit,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_time_limit):
    # Prefix lengths can differ after augment/cut, so decode per-length bucket.
    len_to_indices = defaultdict(list)
    for idx, toks in enumerate(prefix_tokens):
        len_to_indices[len(toks)].append(idx)

    result = []
    for _, indices in sorted(len_to_indices.items(), key=lambda x: x[0], reverse=True):
        bucket = [prefix_tokens[i] for i in indices]
        input_ids = torch.tensor(bucket, device=model.device, dtype=torch.long)
        outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
        suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens,
            max_score=max_score,
            scores=[0.0] * input_ids.size(0),
            pos=input_ids.size(1),
            cache=outputs.past_key_values,
            start_time=time.time(),
            end_time=end_time,
            dfs_time_limit=dfs_time_limit,
        )
        for local_batch_id, beams in suffixes.items():
            global_batch_id = indices[local_batch_id]
            sorted_beams = sorted(beams, key=lambda x:x[0])
            result.append((global_batch_id, sorted_beams))
    result = sorted(result, key=lambda x: x[0])
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    model_name = env_str("ARC_MODEL_NAME", "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    max_seq_length = env_int("ARC_MAX_SEQ_LENGTH", 8192)
    train_aug_n = max(1, env_int("ARC_TRAIN_AUG_N", 16))
    eval_aug_n = max(1, env_int("ARC_EVAL_AUG_N", 2))
    dfs_time_limit = env_float("ARC_DFS_TIME_LIMIT", 540.0)
    min_token_prob = min(max(env_float("ARC_MIN_TOKEN_PROB", 0.2), 1e-6), 1 - 1e-6)
    budget_mode = env_str("ARC_BUDGET_MODE", "adaptive_parallel").strip().lower()
    max_puzzle_seconds = env_float("ARC_MAX_PUZZLE_SECONDS", 1200.0)
    min_puzzle_seconds = env_float("ARC_MIN_PUZZLE_SECONDS", 420.0)
    world_size = max(1, env_int("ARC_WORLD_SIZE", env_int("ARC_NPROCS", 4)))
    augment_mode = env_str("ARC_AUGMENT_MODE", "default").strip().lower()
    train_data_seed = env_int("ARC_TRAIN_DATA_SEED", 1)
    score_seed_offset = env_int("ARC_SCORE_SEED_OFFSET", 0)
    force_all_rounds = env_bool("ARC_FORCE_ALL_ROUNDS", False)
    rescue_aggressive = env_bool("ARC_RESCUE_AGGRESSIVE", False)
    round1_max_fraction = env_float("ARC_ROUND1_MAX_FRACTION", 0.72)
    round1_min_missing_ratio = env_float("ARC_ROUND1_MIN_MISSING_RATIO", 0.08)
    round1_require_incomplete = env_bool("ARC_ROUND1_REQUIRE_INCOMPLETE", True)
    max_scored_beams_round0 = max(0, env_int("ARC_MAX_SCORED_BEAMS_ROUND0", 0))
    max_scored_beams_roundn = max(0, env_int("ARC_MAX_SCORED_BEAMS_ROUNDN", 12))

    eval_seed_list = parse_int_list(os.getenv("ARC_EVAL_SEEDS"), "2")
    eval_aug_n_list = parse_int_list(os.getenv("ARC_EVAL_AUG_NS"), str(eval_aug_n))
    min_token_prob_list = parse_float_list(os.getenv("ARC_MIN_TOKEN_PROBS"), str(min_token_prob))
    dfs_time_limit_list = parse_float_list(os.getenv("ARC_DFS_TIME_LIMITS"), str(dfs_time_limit))

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for _, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = env_str("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
    run_tag = env_str("ARC_RUN_TAG", "").strip()
    os.makedirs(dir_outputs, exist_ok=True)

    print(
        f"[Rank {rank}] budget_mode={budget_mode}, world_size={world_size}, "
        f"train_aug_n={train_aug_n}, eval_rounds={len(eval_seed_list)}, "
        f"base_eval_aug_n={eval_aug_n}, base_dfs_limit={dfs_time_limit:.0f}s, "
        f"round1_max_fraction={round1_max_fraction:.2f}, run_tag='{run_tag}', "
        f"augment_mode='{augment_mode}'"
    )

    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()
        pending_after_pop = get_queue_size(queue)
        remaining_global_time = max(1.0, end_time - start_time)
        puzzle_budget = compute_puzzle_budget(
            budget_mode=budget_mode,
            remaining_global_time=remaining_global_time,
            pending_after_pop=pending_after_pop,
            max_puzzle_seconds=max_puzzle_seconds,
            min_puzzle_seconds=min_puzzle_seconds,
            world_size=world_size,
        )
        puzzle_deadline = min(end_time, start_time + puzzle_budget)

        torch.cuda.reset_peak_memory_stats()

        set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)
        puzzle_ds = arc_test_set.change_keys([key])

        if augment_mode == "extended":
            train_ds = puzzle_ds.augment_v2(n=train_aug_n, shfl_keys=True, seed=train_data_seed)
        else:
            train_ds = puzzle_ds.augment(n=train_aug_n, shfl_keys=True, seed=train_data_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        torch.cuda.reset_peak_memory_stats()
        print(
            f"[Rank {rank}] training stats for puzzle {key}: {stats}; "
            f"budget={puzzle_budget:.1f}s pending={pending_after_pop}"
        )

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        with torch.inference_mode():
            known_scores = {}
            round0_coverage = None
            for round_id, eval_seed in enumerate(eval_seed_list):
                spend_time = time.time() - start_time
                if spend_time > puzzle_budget or time.time() > puzzle_deadline:
                    print(f"[Rank {rank}] timeout before round {round_id} after {spend_time:.1f}s for puzzle {key}")
                    break

                if round_id > 0 and not force_all_rounds:
                    if (spend_time / max(1.0, puzzle_budget)) > round1_max_fraction:
                        print(
                            f"[Rank {rank}] skip round {round_id} for {key}: "
                            f"time fraction {spend_time / max(1.0, puzzle_budget):.2f} > {round1_max_fraction:.2f}"
                        )
                        break
                    if round1_require_incomplete and round0_coverage is not None:
                        round0_missing = 1.0 - round0_coverage
                        if round0_missing < round1_min_missing_ratio:
                            print(
                                f"[Rank {rank}] skip round {round_id} for {key}: "
                                f"round0 missing {round0_missing:.2f} < {round1_min_missing_ratio:.2f}"
                            )
                            break

                round_eval_aug_n = max(1, int(list_value(eval_aug_n_list, round_id)))
                round_min_token_prob = min(max(float(list_value(min_token_prob_list, round_id)), 1e-6), 1 - 1e-6)
                round_max_score = -np.log(round_min_token_prob)
                round_dfs_time_limit = float(list_value(dfs_time_limit_list, round_id))

                eval_ds = puzzle_ds_multi.augment(n=round_eval_aug_n, seed=eval_seed)
                eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

                test_id_to_subkeys = defaultdict(list)
                for subkey in sorted(eval_ds.keys):
                    test_id = subkey.split(".")[0].split("_")[1]
                    test_id_to_subkeys[test_id].append(subkey)

                batches = []
                for _, subkeys in test_id_to_subkeys.items():
                    batches.extend(build_decode_batches(subkeys))
                round_written_subkeys = set()
                round_total_subkeys = len(eval_ds.keys)

                print(
                    f"[Rank {rank}] round={round_id} eval_seed={eval_seed} "
                    f"eval_aug_n={round_eval_aug_n} min_prob={round_min_token_prob:.3f} "
                    f"dfs_limit={round_dfs_time_limit:.0f}s batches={len(batches)}"
                )

                for subkeys in batches:
                    spend_time = time.time() - start_time
                    if spend_time > puzzle_budget or time.time() > puzzle_deadline:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_result = inference_turbo_dfs(
                        model=model,
                        prefix_tokens=tokens,
                        max_new_tokens=max_new_tokens,
                        max_score=round_max_score,
                        end_time=puzzle_deadline,
                        dfs_time_limit=round_dfs_time_limit,
                    )

                    for subkey_id, scored_beams in dfs_result:
                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []
                        max_scored_beams = max_scored_beams_round0 if round_id == 0 else max_scored_beams_roundn

                        for beam_score, tokens in scored_beams:
                            if max_scored_beams > 0 and len(decoded_result) >= max_scored_beams:
                                break
                            array = formatter.convert_tokens_to_array(tokens)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed_from_key(f"{bk}:{score_seed_offset}"))
                                aug_dataset = aug_dataset.cut_to_len(
                                    formatter=formatter,
                                    name="input",
                                    max_len=max_seq_length-max_new_tokens,
                                )
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])

                                if len(aug_queries) == 0:
                                    augmented_scores = [10.0]
                                else:
                                    split = max(1, len(aug_queries) // 2)
                                    augmented_scores1 = calc_scores(aug_queries[:split], aug_answers[:split], tokenizer, model)
                                    augmented_scores2 = (
                                        calc_scores(aug_queries[split:], aug_answers[split:], tokenizer, model)
                                        if split < len(aug_queries) else []
                                    )
                                    augmented_scores = augmented_scores1 + augmented_scores2

                                known_scores[grid_id] = augmented_scores

                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                                "_round": round_id,
                                "_eval_seed": eval_seed,
                            })

                        if len(decoded_result):
                            round_written_subkeys.add(subkey)
                            tag_part = f".{run_tag}" if run_tag else ""
                            store_key = f"{subkey}{tag_part}.run{round_id}"
                            with bz2.BZ2File(os.path.join(dir_outputs, store_key), "w") as f:
                                pickle.dump(decoded_result, f)

                if round_id == 0:
                    round0_coverage = len(round_written_subkeys) / max(1, round_total_subkeys)
                    print(
                        f"[Rank {rank}] round0 coverage for {key}: "
                        f"{len(round_written_subkeys)}/{round_total_subkeys} = {round0_coverage:.2f}"
                    )

            # Auto-aggressive rescue: when coverage is zero after all rounds, retry
            # with much lower threshold before giving up entirely.
            if rescue_aggressive:
                spend_time = time.time() - start_time
                remaining_budget = puzzle_budget - spend_time
                if round0_coverage is not None and round0_coverage < 0.05 and remaining_budget > 120:
                    aggressive_min_prob = max(0.03, round_min_token_prob * 0.25)
                    aggressive_max_score = -np.log(aggressive_min_prob)
                    aggressive_dfs_limit = min(float(list_value(dfs_time_limit_list, 0)) * 0.6, remaining_budget * 0.5)
                    aggressive_eval_seed = int(list_value(eval_seed_list, 0)) + 99
                    aggressive_eval_aug_n = max(2, int(list_value(eval_aug_n_list, 0)) * 2)

                    print(
                        f"[Rank {rank}] aggressive retry for {key}: "
                        f"min_prob={aggressive_min_prob:.4f} dfs_limit={aggressive_dfs_limit:.0f}s "
                        f"eval_seed={aggressive_eval_seed} eval_aug_n={aggressive_eval_aug_n}"
                    )

                    eval_ds = puzzle_ds_multi.augment(n=aggressive_eval_aug_n, seed=aggressive_eval_seed)
                    eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

                    test_id_to_subkeys = defaultdict(list)
                    for subkey in sorted(eval_ds.keys):
                        test_id = subkey.split(".")[0].split("_")[1]
                        test_id_to_subkeys[test_id].append(subkey)

                    batches = []
                    for _, subkeys in test_id_to_subkeys.items():
                        batches.extend(build_decode_batches(subkeys))

                    for subkeys in batches:
                        spend_time = time.time() - start_time
                        if spend_time > puzzle_budget or time.time() > puzzle_deadline:
                            print(f"[Rank {rank}] aggressive timeout for {key}")
                            break

                        tokens = []
                        for subkey in subkeys:
                            data = eval_ds.get(subkey, formatter)
                            tokens.append(tokenizer.encode(data["input"]))

                        dfs_result = inference_turbo_dfs(
                            model=model,
                            prefix_tokens=tokens,
                            max_new_tokens=max_new_tokens,
                            max_score=aggressive_max_score,
                            end_time=puzzle_deadline,
                            dfs_time_limit=aggressive_dfs_limit,
                        )

                        for subkey_id, scored_beams in dfs_result:
                            subkey = subkeys[subkey_id]
                            bk = subkey.split(".")[0]
                            decoded_result = []
                            for beam_score, tokens in scored_beams:
                                array = formatter.convert_tokens_to_array(tokens)
                                if array is None:
                                    continue
                                solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                                decoded_result.append({
                                    "beam_score": beam_score,
                                    "score_aug": [beam_score],
                                    "solution": solution,
                                    "_round": -1,
                                    "_eval_seed": aggressive_eval_seed,
                                })

                            if len(decoded_result):
                                tag_part = f".{run_tag}" if run_tag else ""
                                store_key = f"{subkey}{tag_part}.run_aggr"
                                with bz2.BZ2File(os.path.join(dir_outputs, store_key), "w") as f:
                                    pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import time
import json
import random
import bz2
import pickle
import torch
import argparse
import torch.multiprocessing as mp
from collections import defaultdict


def env_int(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        return default


def env_str(name, default):
    value = os.getenv(name)
    return default if value is None or value == "" else value


def env_bool(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_float(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return float(value)
    except ValueError:
        return default


def parse_key_list(raw):
    if raw is None or raw.strip() == "":
        return None
    return [x.strip() for x in raw.split(",") if x.strip()]


def _parse_output_key(filename):
    base = filename.split(".")[0]
    if "_" not in base:
        return None
    task_key, maybe_sub = base.rsplit("_", 1)
    if not maybe_sub.isdigit():
        return None
    return task_key, int(maybe_sub), f"{task_key}_{int(maybe_sub)}"


def _solution_key(solution):
    try:
        if hasattr(solution, "tolist"):
            solution = solution.tolist()
        return tuple(tuple(int(x) for x in row) for row in solution)
    except Exception:
        return None


def infer_output_quality(output_dir, data):
    if not os.path.isdir(output_dir):
        return {}

    per_output = defaultdict(lambda: {
        "files": 0,
        "candidates": 0,
        "unique": set(),
        "best_beam": None,
    })

    for filename in sorted(os.listdir(output_dir)):
        parsed = _parse_output_key(filename)
        if parsed is None:
            continue
        _, _, output_key = parsed
        stats = per_output[output_key]
        stats["files"] += 1

        path = os.path.join(output_dir, filename)
        if not os.path.isfile(path):
            continue
        try:
            with bz2.BZ2File(path) as f:
                outputs = pickle.load(f)
        except Exception:
            continue

        if not isinstance(outputs, list):
            continue
        stats["candidates"] += len(outputs)
        for sample in outputs:
            if not isinstance(sample, dict):
                continue
            skey = _solution_key(sample.get("solution"))
            if skey is not None:
                stats["unique"].add(skey)
            try:
                beam = float(sample.get("beam_score"))
            except Exception:
                beam = None
            if beam is not None and (stats["best_beam"] is None or beam < stats["best_beam"]):
                stats["best_beam"] = beam

    quality = {}
    for task_key, task in data.items():
        need = len(task.get("test", []))
        if need <= 0:
            continue
        rows = []
        for i in range(need):
            output_key = f"{task_key}_{i}"
            raw = per_output.get(output_key)
            if raw is None:
                rows.append({"files": 0, "candidates": 0, "unique": 0, "best_beam": None})
            else:
                rows.append({
                    "files": raw["files"],
                    "candidates": raw["candidates"],
                    "unique": len(raw["unique"]),
                    "best_beam": raw["best_beam"],
                })

        files = [r["files"] for r in rows]
        candidates = [r["candidates"] for r in rows]
        unique = [r["unique"] for r in rows]
        beams = [r["best_beam"] for r in rows if r["best_beam"] is not None]
        quality[task_key] = {
            "min_files": min(files) if files else 0,
            "mean_files": (sum(files) / len(files)) if files else 0.0,
            "min_candidates": min(candidates) if candidates else 0,
            "mean_candidates": (sum(candidates) / len(candidates)) if candidates else 0.0,
            "min_unique": min(unique) if unique else 0,
            "mean_unique": (sum(unique) / len(unique)) if unique else 0.0,
            "best_beam": min(beams) if beams else None,
        }
    return quality


def infer_completed_keys(
    output_dir,
    data,
    min_files_per_test=1,
    min_unique_per_test=0,
    min_candidates_per_test=0,
    mode="files",
):
    quality = infer_output_quality(output_dir, data)

    completed = set()
    for task_key, task in data.items():
        need = len(task.get("test", []))
        if need <= 0:
            continue
        task_quality = quality.get(task_key, {})
        ok = True
        if task_quality.get("min_files", 0) < min_files_per_test:
            ok = False
        if mode == "quality":
            if task_quality.get("min_unique", 0) < min_unique_per_test:
                ok = False
            if task_quality.get("min_candidates", 0) < min_candidates_per_test:
                ok = False
        if ok:
            completed.add(task_key)
    return completed


def infer_key_coverage(output_dir, data):
    return infer_output_quality(output_dir, data)


def task_complexity(task):
    total_cells = 0
    total_pairs = len(task.get("train", [])) + len(task.get("test", []))
    for section in ("train", "test"):
        for sample in task.get(section, []):
            inp = sample.get("input", [])
            total_cells += len(inp) * (len(inp[0]) if len(inp) else 0)
            if section == "train":
                out = sample.get("output", [])
                total_cells += len(out) * (len(out[0]) if len(out) else 0)
    return total_cells + 50 * total_pairs


def order_keys(data, mode):
    keys = list(data.keys())
    if mode == "lexicographic":
        return sorted(keys)
    scored = [(k, task_complexity(data[k])) for k in keys]
    if mode == "hard_first":
        scored.sort(key=lambda x: x[1], reverse=True)
        return [k for k, _ in scored]
    if mode == "random":
        rnd = random.Random(env_int("ARC_TASK_ORDER_SEED", 42))
        rnd.shuffle(keys)
        return keys
    scored.sort(key=lambda x: x[1])
    return [k for k, _ in scored]


def parse_gpu_ids():
    raw = parse_key_list(os.getenv("ARC_GPU_IDS"))
    if raw is not None:
        gpu_ids = [int(x) for x in raw]
    else:
        gpu_count = torch.cuda.device_count()
        gpu_ids = list(range(gpu_count))
    if len(gpu_ids) == 0:
        return [0]
    return gpu_ids


def worker_marker_path(marker_tag, rank):
    return f"/kaggle/worker_{marker_tag}_{rank}"


def clear_worker_markers(marker_tag, nprocs):
    for r in range(nprocs):
        marker = worker_marker_path(marker_tag, r)
        if os.path.exists(marker):
            os.remove(marker)


def local_worker(rank, queue, end_time, gpu_ids, marker_tag):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_ids[rank])
    torch.set_default_device("cpu")

    # Unsloth patching conflict workaround.
    if rank > 0:
        prev_marker = worker_marker_path(marker_tag, rank - 1)
        while not os.path.exists(prev_marker):
            time.sleep(5)

    from arc_solver import worker

    current_marker = worker_marker_path(marker_tag, rank)
    with open(current_marker, "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")
    worker(rank, queue, end_time)
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    os.environ.setdefault("PYTHONHASHSEED", str(env_int("ARC_PYTHONHASHSEED", 42)))

    task_order = env_str("ARC_TASK_ORDER", "lexicographic").strip().lower()
    keys = order_keys(data, task_order)

    key_filter = parse_key_list(os.getenv("ARC_KEYS"))
    if key_filter is not None:
        allowed = set(key_filter)
        keys = [k for k in keys if k in allowed]

    eval_default_keys = ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]
    eval_keys = parse_key_list(os.getenv("ARC_EVAL_KEYS"))
    use_default_eval_subset = env_bool("ARC_USE_DEFAULT_EVAL_SUBSET", True)
    if not rerun_mode and use_default_eval_subset:
        keep = set(eval_default_keys if eval_keys is None else eval_keys)
        keys = [k for k in keys if k in keep]

    task_limit = env_int("ARC_TASK_LIMIT", 0)
    if task_limit > 0:
        keys = keys[:task_limit]

    skip_done = env_bool("ARC_SKIP_DONE", False)
    output_dir = env_str("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
    skip_done_mode = env_str("ARC_SKIP_DONE_MODE", "files").strip().lower()
    skip_done_min_files_per_test = max(1, env_int("ARC_SKIP_DONE_MIN_FILES_PER_TEST", 1))
    skip_done_min_unique_per_test = max(0, env_int("ARC_SKIP_DONE_MIN_UNIQUE_PER_TEST", 0))
    skip_done_min_candidates_per_test = max(0, env_int("ARC_SKIP_DONE_MIN_CANDIDATES_PER_TEST", 0))
    if skip_done:
        done_keys = infer_completed_keys(
            output_dir,
            data,
            min_files_per_test=skip_done_min_files_per_test,
            min_unique_per_test=skip_done_min_unique_per_test,
            min_candidates_per_test=skip_done_min_candidates_per_test,
            mode=skip_done_mode,
        )
        before = len(keys)
        keys = [k for k in keys if k not in done_keys]
        print(
            f"[Starter] skip_done={skip_done}, output_dir='{output_dir}', "
            f"mode={skip_done_mode}, "
            f"min_files_per_test={skip_done_min_files_per_test}, "
            f"min_unique_per_test={skip_done_min_unique_per_test}, "
            f"min_candidates_per_test={skip_done_min_candidates_per_test}, "
            f"done={before - len(keys)}, remaining={len(keys)}"
        )

    # Optional hard-focus pass: keep only the least-covered tasks among remaining keys.
    focus_hard_n = max(0, env_int("ARC_FOCUS_HARD_N", 0))
    if focus_hard_n > 0 and len(keys) > focus_hard_n:
        coverage = infer_key_coverage(output_dir, data)
        focus_mode = env_str("ARC_FOCUS_MODE", "files").strip().lower()
        # Quality mode prioritizes outputs with low diversity/candidate count, not just low file count.
        keys_sorted = sorted(
            keys,
            key=(
                (lambda k: (
                    coverage.get(k, {}).get("min_unique", 0),
                    coverage.get(k, {}).get("min_candidates", 0),
                    coverage.get(k, {}).get("min_files", 0),
                    coverage.get(k, {}).get("mean_unique", 0.0),
                    coverage.get(k, {}).get("mean_candidates", 0.0),
                    coverage.get(k, {}).get("mean_files", 0.0),
                    -task_complexity(data[k]),
                    k,
                ))
                if focus_mode == "quality"
                else (lambda k: (
                    coverage.get(k, {}).get("min_files", 0),
                    coverage.get(k, {}).get("mean_files", 0.0),
                    -task_complexity(data[k]),
                    k,
                ))
            ),
        )
        keys = keys_sorted[:focus_hard_n]
        print(
            f"[Starter] focus_hard_n={focus_hard_n}, focus_mode={focus_mode}, "
            f"selected={len(keys)} least-covered/lowest-quality tasks"
        )
        for k in keys[:12]:
            q = coverage.get(k, {})
            print(
                f"[Starter] focus {k}: files={q.get('min_files', 0)}/"
                f"{q.get('mean_files', 0.0):.1f}, unique={q.get('min_unique', 0)}/"
                f"{q.get('mean_unique', 0.0):.1f}, cand={q.get('min_candidates', 0)}/"
                f"{q.get('mean_candidates', 0.0):.1f}"
            )

    gpu_ids = parse_gpu_ids()
    max_procs = len(gpu_ids)
    requested_procs = env_int("ARC_NPROCS", max_procs)
    # Do not spawn more processes than pending tasks:
    # extra ranks still pay full model load cost before they can exit.
    nprocs = max(1, min(requested_procs, max_procs, len(keys)))
    gpu_ids = gpu_ids[:nprocs]

    os.environ.setdefault("ARC_WORLD_SIZE", str(nprocs))

    print(
        f"[Starter] tasks={len(keys)}, order={task_order}, "
        f"nprocs={nprocs}, gpu_ids={gpu_ids}, rerun={bool(rerun_mode)}"
    )

    if len(keys) == 0:
        print("[Starter] no tasks to run, exiting without spawning workers")
        raise SystemExit(0)

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    marker_tag = env_str("ARC_WORKER_MARKER_TAG", f"{int(time.time())}_{os.getpid()}")
    clear_worker_markers(marker_tag, nprocs)
    print(f"[Starter] worker_marker_tag={marker_tag}")

    mp.spawn(local_worker, args=(queue, args.end_time, gpu_ids, marker_tag), nprocs=nprocs)


In [ ]:
%%writefile build_submission.py
import os
import json
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import (
    ArcDecoder,
    score_kgmon,
    score_full_probmul_3,
    score_hybrid_consensus,
    score_round_consensus,
)


def env_int(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        return default


def parse_dirs(raw, default):
    if raw is None or raw.strip() == "":
        return [default]
    return [x.strip() for x in raw.split(",") if x.strip()]


def write_validation_report(path, split_data, decoder, selected_results):
    rows = []
    for key in split_data.keys:
        label = split_data.replies.get(key, [None])[0]
        decoded = decoder.decoded_results.get(key, {})
        candidates = list(decoded.values())
        selected = selected_results.get(key, [])

        correct_candidates = []
        if label is not None:
            for sample in candidates:
                if np.array_equal(sample["solution"], label):
                    correct_candidates.append(sample)

        attempt_correct = []
        if label is not None:
            attempt_correct = [bool(np.array_equal(guess, label)) for guess in selected[:2]]
        while len(attempt_correct) < 2:
            attempt_correct.append(False)

        # Keep the hashing explicit; candidate grids are small and this is report-only.
        unique_candidates = set()
        for sample in candidates:
            try:
                unique_candidates.add(tuple(tuple(int(x) for x in row) for row in np.asarray(sample["solution"]).tolist()))
            except Exception:
                pass

        rows.append({
            "key": key,
            "num_candidates": len(candidates),
            "num_unique_candidates": len(unique_candidates),
            "candidate_hit": len(correct_candidates) > 0,
            "attempt_1_correct": attempt_correct[0],
            "attempt_2_correct": attempt_correct[1],
            "pass2_correct": any(attempt_correct),
            "best_correct_beam": (
                min(float(sample["beam_score"]) for sample in correct_candidates)
                if correct_candidates else None
            ),
        })

    summary = {
        "outputs": len(rows),
        "candidate_hit": sum(1 for row in rows if row["candidate_hit"]),
        "pass2_correct": sum(1 for row in rows if row["pass2_correct"]),
        "attempt_1_correct": sum(1 for row in rows if row["attempt_1_correct"]),
        "attempt_2_correct": sum(1 for row in rows if row["attempt_2_correct"]),
        "zero_candidate_outputs": sum(1 for row in rows if row["num_candidates"] == 0),
        "rows": rows,
    }
    with open(path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"*** Validation report saved to '{path}'")
    print(
        "*** Validation report summary:",
        f"pass2={summary['pass2_correct']}/{summary['outputs']}",
        f"candidate_hit={summary['candidate_hit']}/{summary['outputs']}",
        f"zero_candidates={summary['zero_candidate_outputs']}",
    )


def main():
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    n_guesses = max(1, env_int("ARC_N_GUESSES", 2))
    inference_dirs = parse_dirs(os.getenv("ARC_INFERENCE_DIRS"), "/kaggle/inference_outputs")
    output_path = os.getenv("ARC_SUBMISSION_PATH", "/kaggle/working/submission.json")
    selection_mode = os.getenv("ARC_SELECTION_MODE", "kgmon").strip().lower()

    if rerun_mode:
        data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
    else:
        data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
        data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

    split_data = data.split_multi_replies()
    decoder = ArcDecoder(split_data, n_guesses=n_guesses)

    loaded = 0
    for i, store in enumerate(inference_dirs):
        if not os.path.isdir(store):
            print(f"[WARN] Skip missing inference dir: {store}")
            continue
        run_name = f".run{i}"
        print(f"*** Loading decoded outputs from '{store}' ({run_name})")
        decoder.load_decoded_results(store, run_name=run_name)
        loaded += 1

    if loaded == 0:
        raise RuntimeError("No inference outputs loaded. Check ARC_INFERENCE_DIRS.")
    if len(decoder.decoded_results) == 0:
        raise RuntimeError("Inference directory exists but no decoded result files were parsed.")

    if selection_mode == "kgmon":
        selected_results = decoder.run_selection_algo(score_kgmon)
    elif selection_mode == "full":
        selected_results = decoder.run_selection_algo(score_full_probmul_3)
    elif selection_mode == "hybrid":
        selected_results = decoder.run_selection_algo(score_hybrid_consensus)
    elif selection_mode == "round":
        selected_results = decoder.run_selection_algo(score_round_consensus)
    elif selection_mode == "ensemble":
        selected_results = decoder.run_selection_ensemble()
    elif selection_mode == "dual":
        selected_results = decoder.run_selection_dual()
    elif selection_mode == "dual_round":
        selected_results = decoder.run_selection_dual_round()
    elif selection_mode == "guarded_dual":
        selected_results = decoder.run_selection_guarded_dual()
    elif selection_mode == "diverse_dual":
        selected_results = decoder.run_selection_diverse_dual()
    else:
        print(f"[WARN] Unknown ARC_SELECTION_MODE='{selection_mode}', fallback to kgmon")
        selected_results = decoder.run_selection_algo(score_kgmon)

    submission = data.get_submission(selected_results)

    with open(output_path, "w") as f:
        json.dump(submission, f)
    print(f"*** Submission saved to '{output_path}' (selection_mode={selection_mode})")

    if not rerun_mode:
        report_path = os.getenv("ARC_REPORT_PATH", "").strip()
        if report_path:
            write_validation_report(report_path, split_data, decoder, selected_results)
        decoder.benchmark_selection_algos()
        with open(output_path, "r") as f:
            reload_submission = json.load(f)
        print("*** Reload score:", data.validate_submission(reload_submission))


if __name__ == "__main__":
    main()


In [ ]:
import os, shutil

# Fresh outputs avoid mixing old local/debug files with this submission run.
shutil.rmtree("/kaggle/inference_outputs_main", ignore_errors=True)
os.makedirs("/kaggle/inference_outputs_main", exist_ok=True)

# Stage-1: original-strength baseline. This is the score anchor.
# Keep train_aug=16, eval_aug=2, min_prob=0.20, dfs=540, no beam truncation.
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 ARC_OUTPUT_DIR=/kaggle/inference_outputs_main ARC_RUN_TAG=base ARC_SKIP_DONE=0 ARC_TASK_ORDER=lexicographic ARC_BUDGET_MODE=fixed ARC_MAX_PUZZLE_SECONDS=1200 ARC_MIN_PUZZLE_SECONDS=420 ARC_NPROCS=4 ARC_TRAIN_AUG_N=16 ARC_TRAIN_DATA_SEED=1 ARC_EVAL_SEEDS=2 ARC_EVAL_AUG_NS=2 ARC_MIN_TOKEN_PROBS=0.20 ARC_DFS_TIME_LIMITS=540 ARC_MAX_SCORED_BEAMS_ROUND0=0 ARC_MAX_SCORED_BEAMS_ROUNDN=0 ARC_SCORE_SEED_OFFSET=13 ARC_WORKER_MARKER_TAG=base python starter.py --end-time {global_end_time}

# Stage-2: coverage rescue only. Unique=1 is accepted because a stable single grid can be the correct ARC answer.
# This fixes v8's wasteful repeated reruns on high-confidence single-answer tasks.
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 ARC_OUTPUT_DIR=/kaggle/inference_outputs_main ARC_RUN_TAG=rescue_a ARC_SKIP_DONE=1 ARC_SKIP_DONE_MODE=quality ARC_SKIP_DONE_MIN_FILES_PER_TEST=10 ARC_SKIP_DONE_MIN_UNIQUE_PER_TEST=1 ARC_SKIP_DONE_MIN_CANDIDATES_PER_TEST=6 ARC_FOCUS_MODE=quality ARC_FOCUS_HARD_N=16 ARC_TASK_ORDER=hard_first ARC_BUDGET_MODE=adaptive_parallel ARC_MAX_PUZZLE_SECONDS=840 ARC_MIN_PUZZLE_SECONDS=220 ARC_NPROCS=4 ARC_AUGMENT_MODE=extended ARC_TRAIN_AUG_N=8 ARC_TRAIN_DATA_SEED=7 ARC_EVAL_SEEDS=23 ARC_EVAL_AUG_NS=1 ARC_MIN_TOKEN_PROBS=0.16 ARC_DFS_TIME_LIMITS=190 ARC_ROUND1_REQUIRE_INCOMPLETE=1 ARC_MAX_SCORED_BEAMS_ROUND0=10 ARC_MAX_SCORED_BEAMS_ROUNDN=8 ARC_SCORE_SEED_OFFSET=17 ARC_WORKER_MARKER_TAG=rescue_a python starter.py --end-time {global_end_time}

# Stage-3: narrow last pass for genuinely low-coverage outputs, not low-unique outputs.
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 ARC_OUTPUT_DIR=/kaggle/inference_outputs_main ARC_RUN_TAG=rescue_b ARC_SKIP_DONE=1 ARC_SKIP_DONE_MODE=quality ARC_SKIP_DONE_MIN_FILES_PER_TEST=14 ARC_SKIP_DONE_MIN_UNIQUE_PER_TEST=1 ARC_SKIP_DONE_MIN_CANDIDATES_PER_TEST=8 ARC_FOCUS_MODE=quality ARC_FOCUS_HARD_N=8 ARC_TASK_ORDER=hard_first ARC_BUDGET_MODE=adaptive_parallel ARC_MAX_PUZZLE_SECONDS=700 ARC_MIN_PUZZLE_SECONDS=200 ARC_NPROCS=4 ARC_AUGMENT_MODE=extended ARC_RESCUE_AGGRESSIVE=1 ARC_TRAIN_AUG_N=8 ARC_TRAIN_DATA_SEED=11 ARC_EVAL_SEEDS=37 ARC_EVAL_AUG_NS=1 ARC_MIN_TOKEN_PROBS=0.13 ARC_DFS_TIME_LIMITS=130 ARC_MAX_SCORED_BEAMS_ROUND0=8 ARC_SCORE_SEED_OFFSET=23 ARC_WORKER_MARKER_TAG=rescue_b python starter.py --end-time {global_end_time}

# Guarded final selection:
# attempt_1 = kgmon anchored to baseline candidates when available;
# attempt_2 = best distinct full-probability candidate from all candidates.
!ARC_INFERENCE_DIRS=/kaggle/inference_outputs_main ARC_SELECTION_MODE=diverse_dual ARC_SUBMISSION_PATH=/kaggle/working/submission.json python build_submission.py
